# Masked Face Verification Validation Spike

This notebook runs a small masked-face verification spike on a real masked/unmasked identity dataset such as PKU-Masked-Face, or RMFRD/SMFRD as a fallback. It compares a full-face baseline, an upper-face occlusion-aware variant, and a second pretrained recognizer candidate. Outputs are saved under `results/`.

## Runtime Setup

In Colab, run this cell first. It installs only missing optional packages and does not replace Colab's preinstalled `torch`, `numpy`, `opencv`, or other core runtime packages.

In [ ]:
import importlib.util
import subprocess
import sys

core_modules = ["cv2", "matplotlib", "numpy", "pandas", "PIL", "sklearn", "torch", "torchvision"]
missing_core = [module for module in core_modules if importlib.util.find_spec(module) is None]
if missing_core:
    raise RuntimeError(
        "Missing expected Colab runtime packages: "
        + ", ".join(missing_core)
        + ". Use a standard Colab runtime instead of pip-replacing core dependencies."
    )

optional = {
    "facenet_pytorch": "facenet-pytorch==2.6.0",
    "gdown": "gdown>=5.2.0",
    "seaborn": "seaborn>=0.13.0",
    "tqdm": "tqdm>=4.66.0",
}
installed = []
for module, package in optional.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", package])
        installed.append(package)

print("Installed optional packages without dependencies:" if installed else "Dependencies already available", installed)

## Configuration

Place the dataset in Drive or upload an archive. Preferred normalized layout:

```text
data/pku_masked_face_subset/person_001/masked/*.jpg
data/pku_masked_face_subset/person_001/unmasked/*.jpg
```

The loader also handles common two-root layouts where masked and unmasked roots contain identity subfolders.

In [ ]:
from pathlib import Path

# Optional: set to a direct URL for a dataset archive. Leave empty when using Drive/uploaded files.
DATASET_ARCHIVE_URL = ""
DATASET_ARCHIVE_PATH = Path("data/dataset_archive")

# Point this to the normalized dataset root or to a root containing masked/unmasked folders.
DATA_ROOT = Path("data/pku_masked_face_subset")

# Keep this small for the first spike pass.
MAX_IDENTITIES = 150
PAIRS_PER_CASE = 600
IMAGE_SIZE = 160
RANDOM_SEED = 42
RESULTS_DIR = Path("results")
QUAL_DIR = RESULTS_DIR / "qualitative_examples"

# Optional third-model hook. The spike can run with the CASIA FaceNet candidate immediately,
# but the intended masked-specific slot is a checkpoint such as MaskInv KD / ElasticFace-Arc.
# Reference: https://github.com/fdbtrs/Masked-Face-Recognition-KD
MASKED_SPECIFIC_MODEL_PATH = Path("models/masked_specific_model.pt")
MASKED_SPECIFIC_MODEL_NOTE = "default candidate is FaceNet CASIA-WebFace; replace with MaskInv KD / ElasticFace-Arc checkpoint when available"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
QUAL_DIR.mkdir(parents=True, exist_ok=True)
print("Configured", {"data_root": str(DATA_ROOT), "max_identities": MAX_IDENTITIES, "pairs_per_case": PAIRS_PER_CASE})

In [ ]:
import hashlib
import itertools
import math
import os
import random
import shutil
import zipfile
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from facenet_pytorch import InceptionResnetV1, MTCNN
from PIL import Image, ImageDraw
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## Dataset Preparation

If needed, download or extract a small benchmark subset before discovery.

In [ ]:
def maybe_download_archive(url: str, dest: Path) -> Optional[Path]:
    if not url:
        return None
    dest.parent.mkdir(parents=True, exist_ok=True)
    suffix = ".zip" if ".zip" in url.lower() else ""
    out = dest.with_suffix(suffix)
    if out.exists():
        return out
    if "drive.google.com" in url:
        import gdown
        gdown.download(url=url, output=str(out), fuzzy=True, quiet=False)
    else:
        import urllib.request
        urllib.request.urlretrieve(url, out)
    return out


def maybe_extract_archive(archive: Optional[Path], target: Path) -> None:
    if archive is None or target.exists():
        return
    target.mkdir(parents=True, exist_ok=True)
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(target)
    else:
        shutil.unpack_archive(str(archive), str(target))


archive = maybe_download_archive(DATASET_ARCHIVE_URL, DATASET_ARCHIVE_PATH)
maybe_extract_archive(archive, DATA_ROOT)
print("Dataset root exists:", DATA_ROOT.exists(), DATA_ROOT)

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


@dataclass(frozen=True)
class FaceRecord:
    identity: str
    condition: str
    path: Path


def image_files(root: Path) -> List[Path]:
    if not root.exists():
        return []
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS and p.is_file())


def normalize_condition(name: str) -> Optional[str]:
    text = name.lower().replace("-", "_").replace(" ", "_")
    unmasked_tokens = [
        "unmasked", "without_mask", "without_masks", "no_mask", "no_masks",
        "non_mask", "non_masked", "nomask", "common", "normal", "holistic",
        "afdb_face_dataset", "face_dataset",
    ]
    masked_tokens = [
        "masked", "with_mask", "with_masks", "mask", "rmfrd", "smfrd",
        "afdb_masked_face_dataset", "masked_face_dataset",
    ]
    if any(token in text for token in masked_tokens):
        return "masked"
    if any(token in text for token in unmasked_tokens):
        return "unmasked"
    return None


def discover_identity_condition_layout(root: Path) -> List[FaceRecord]:
    records: List[FaceRecord] = []
    for identity_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        for condition_dir in sorted(p for p in identity_dir.iterdir() if p.is_dir()):
            condition = normalize_condition(condition_dir.name)
            if condition:
                records.extend(FaceRecord(identity_dir.name, condition, img) for img in image_files(condition_dir))
    return records


def discover_condition_identity_layout(root: Path) -> List[FaceRecord]:
    records: List[FaceRecord] = []
    for condition_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        condition = normalize_condition(condition_dir.name)
        if not condition:
            continue
        identity_dirs = [p for p in condition_dir.iterdir() if p.is_dir()]
        if identity_dirs:
            for identity_dir in sorted(identity_dirs):
                records.extend(FaceRecord(identity_dir.name, condition, img) for img in image_files(identity_dir))
        else:
            for img in image_files(condition_dir):
                identity = img.stem.split("_")[0].split("-")[0]
                records.append(FaceRecord(identity, condition, img))
    return records


def load_records(root: Path, max_identities: int) -> List[FaceRecord]:
    if not root.exists():
        raise FileNotFoundError(f"DATA_ROOT does not exist: {root}. Mount Drive, upload data, or set DATASET_ARCHIVE_URL.")
    candidates = [discover_identity_condition_layout(root), discover_condition_identity_layout(root)]
    records = max(candidates, key=len)
    by_id: Dict[str, Dict[str, List[FaceRecord]]] = {}
    for rec in records:
        by_id.setdefault(rec.identity, {}).setdefault(rec.condition, []).append(rec)
    valid_ids = [ident for ident, groups in by_id.items() if groups.get("masked") and groups.get("unmasked")]
    random.shuffle(valid_ids)
    selected = set(valid_ids[:max_identities])
    subset = [rec for rec in records if rec.identity in selected]
    if len(selected) < 2:
        raise ValueError("Need at least two identities with both masked and unmasked images.")
    print(f"Selected {len(selected)} identities and {len(subset)} images from {root}")
    return subset


records = load_records(DATA_ROOT, MAX_IDENTITIES)
pd.DataFrame([r.__dict__ for r in records]).groupby(["condition"]).size()

## Model Variants

The third variant is intentionally isolated so a true masked-face-specific checkpoint can replace the candidate model without changing the rest of the evaluation.

In [ ]:
class FaceEmbedder:
    def __init__(self, pretrained: str, variant: str):
        self.pretrained = pretrained
        self.variant = variant
        self.mtcnn = MTCNN(image_size=IMAGE_SIZE, margin=16, post_process=True, device=DEVICE)
        self.model = InceptionResnetV1(pretrained=pretrained).eval().to(DEVICE)

    def _upper_face(self, face_tensor: torch.Tensor) -> torch.Tensor:
        _, h, w = face_tensor.shape
        crop_h = int(h * 0.62)
        upper = face_tensor[:, :crop_h, :].unsqueeze(0)
        resized = torch.nn.functional.interpolate(upper, size=(h, w), mode="bilinear", align_corners=False)[0]
        return resized

    def _lower_blackout(self, face_tensor: torch.Tensor) -> torch.Tensor:
        _, h, _ = face_tensor.shape
        start = int(h * 0.55)
        out = face_tensor.clone()
        out[:, start:, :] = 0.0
        return out

    def _lower_blur(self, face_tensor: torch.Tensor) -> torch.Tensor:
        _, h, _ = face_tensor.shape
        start = int(h * 0.55)
        out = face_tensor.clone()
        lower = out[:, start:, :].unsqueeze(0)
        blurred = torch.nn.functional.avg_pool2d(lower, kernel_size=15, stride=1, padding=7)
        out[:, start:, :] = blurred[0, :, : h - start, :]
        return out

    @torch.inference_mode()
    def embed_one(self, path: Path) -> Optional[np.ndarray]:
        try:
            img = Image.open(path).convert("RGB")
            face = self.mtcnn(img)
        except Exception:
            return None
        if face is None:
            return None
        if self.variant == "upper_face":
            face = self._upper_face(face)
        elif self.variant == "lower_blackout":
            face = self._lower_blackout(face)
        elif self.variant == "lower_blur":
            face = self._lower_blur(face)
        batch = face.unsqueeze(0).to(DEVICE)
        emb = self.model(batch).detach().cpu().numpy()[0]
        norm = np.linalg.norm(emb)
        return emb / norm if norm else emb


def load_models() -> Dict[str, FaceEmbedder]:
    return {
        "baseline_facenet_vggface2": FaceEmbedder("vggface2", "full_face"),
        "upper_face_facenet_vggface2": FaceEmbedder("vggface2", "upper_face"),
        "lower_blackout_facenet_vggface2": FaceEmbedder("vggface2", "lower_blackout"),
        "lower_blur_facenet_vggface2": FaceEmbedder("vggface2", "lower_blur"),
        "masked_specific_candidate_facenet_casia_webface": FaceEmbedder("casia-webface", "full_face"),
    }


models = load_models()
print("Models:", list(models), "masked_candidate_note=", MASKED_SPECIFIC_MODEL_NOTE)

In [ ]:
def compute_embeddings(paths: Sequence[Path], models: Dict[str, FaceEmbedder]) -> Dict[str, Dict[Path, np.ndarray]]:
    embeddings: Dict[str, Dict[Path, np.ndarray]] = {name: {} for name in models}
    for model_name, embedder in models.items():
        failures = 0
        for path in tqdm(paths, desc=f"Embedding {model_name}"):
            emb = embedder.embed_one(path)
            if emb is None:
                failures += 1
            else:
                embeddings[model_name][path] = emb
        print(model_name, "embedded", len(embeddings[model_name]), "failed", failures)
    return embeddings

## Verification Pairs and Metrics

In [ ]:
@dataclass(frozen=True)
class Pair:
    case: str
    label: int
    left: Path
    right: Path
    left_id: str
    right_id: str


def records_by_identity(records: Sequence[FaceRecord]) -> Dict[str, Dict[str, List[FaceRecord]]]:
    grouped: Dict[str, Dict[str, List[FaceRecord]]] = {}
    for rec in records:
        grouped.setdefault(rec.identity, {}).setdefault(rec.condition, []).append(rec)
    return grouped


def sample_same_identity_pairs(grouped: Dict[str, Dict[str, List[FaceRecord]]], case: str, n: int) -> List[Pair]:
    left_cond, right_cond = case.split("-")
    ids = [ident for ident, groups in grouped.items() if groups.get(left_cond) and groups.get(right_cond)]
    pairs: List[Pair] = []
    attempts = 0
    while len(pairs) < n and attempts < n * 20:
        attempts += 1
        ident = random.choice(ids)
        left = random.choice(grouped[ident][left_cond])
        right = random.choice(grouped[ident][right_cond])
        if left.path != right.path:
            pairs.append(Pair(case, 1, left.path, right.path, ident, ident))
    return pairs


def sample_different_identity_pairs(grouped: Dict[str, Dict[str, List[FaceRecord]]], case: str, n: int) -> List[Pair]:
    left_cond, right_cond = case.split("-")
    ids = [ident for ident, groups in grouped.items() if groups.get(left_cond) and groups.get(right_cond)]
    pairs: List[Pair] = []
    attempts = 0
    while len(pairs) < n and attempts < n * 20:
        attempts += 1
        left_id, right_id = random.sample(ids, 2)
        left = random.choice(grouped[left_id][left_cond])
        right = random.choice(grouped[right_id][right_cond])
        pairs.append(Pair(case, 0, left.path, right.path, left_id, right_id))
    return pairs


def build_pairs(records: Sequence[FaceRecord], pairs_per_case: int) -> List[Pair]:
    grouped = records_by_identity(records)
    cases = ["masked-masked", "masked-unmasked", "unmasked-unmasked"]
    pairs: List[Pair] = []
    for case in cases:
        positives = sample_same_identity_pairs(grouped, case, pairs_per_case // 2)
        negatives = sample_different_identity_pairs(grouped, case, pairs_per_case // 2)
        pairs.extend(positives + negatives)
    random.shuffle(pairs)
    return pairs


pairs = build_pairs(records, PAIRS_PER_CASE)
pair_paths = sorted({path for pair in pairs for path in (pair.left, pair.right)})
print("pairs", len(pairs), "unique_eval_images", len(pair_paths))
display(pd.DataFrame([p.__dict__ for p in pairs]).groupby(["case", "label"]).size())
embeddings_by_model = compute_embeddings(pair_paths, models)

In [ ]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def best_threshold_metrics(labels: np.ndarray, scores: np.ndarray) -> Dict[str, float]:
    thresholds = np.unique(scores)
    best = {"accuracy": -1.0, "threshold": float(thresholds[0]), "far": math.nan, "frr": math.nan}
    for threshold in thresholds:
        preds = (scores >= threshold).astype(int)
        acc = accuracy_score(labels, preds)
        fp = int(((preds == 1) & (labels == 0)).sum())
        tn = int(((preds == 0) & (labels == 0)).sum())
        fn = int(((preds == 0) & (labels == 1)).sum())
        tp = int(((preds == 1) & (labels == 1)).sum())
        far = fp / max(fp + tn, 1)
        frr = fn / max(fn + tp, 1)
        if acc > best["accuracy"]:
            best = {"accuracy": float(acc), "threshold": float(threshold), "far": float(far), "frr": float(frr)}
    return best


def evaluate_model(model_name: str, embeddings: Dict[Path, np.ndarray], pairs: Sequence[Pair]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    skipped = 0
    for pair in pairs:
        if pair.left not in embeddings or pair.right not in embeddings:
            skipped += 1
            continue
        rows.append({
            "model": model_name,
            "case": pair.case,
            "label": pair.label,
            "score": cosine(embeddings[pair.left], embeddings[pair.right]),
            "left": str(pair.left),
            "right": str(pair.right),
            "left_id": pair.left_id,
            "right_id": pair.right_id,
        })
    scores_df = pd.DataFrame(rows)
    metrics = []
    for case, case_df in scores_df.groupby("case"):
        labels = case_df["label"].to_numpy()
        scores = case_df["score"].to_numpy()
        auc = roc_auc_score(labels, scores) if len(np.unique(labels)) == 2 else math.nan
        threshold_metrics = best_threshold_metrics(labels, scores)
        metrics.append({
            "model": model_name,
            "case": case,
            "pairs": len(case_df),
            "skipped_pairs_total": skipped,
            "roc_auc": float(auc),
            **threshold_metrics,
        })
    return scores_df, pd.DataFrame(metrics)


all_scores = []
all_metrics = []
for model_name, embeddings in embeddings_by_model.items():
    scores_df, metrics_df = evaluate_model(model_name, embeddings, pairs)
    all_scores.append(scores_df)
    all_metrics.append(metrics_df)

scores = pd.concat(all_scores, ignore_index=True)
metrics = pd.concat(all_metrics, ignore_index=True).sort_values(["case", "model"])
scores.to_csv(RESULTS_DIR / "validation_pair_scores.csv", index=False)
metrics.to_csv(RESULTS_DIR / "validation_results.csv", index=False)
metrics

## Lightweight Fusion Feasibility

Tune a single score-fusion weight on masked-unmasked calibration pairs, then evaluate on held-out pairs for all cases. This tests whether lightweight occlusion handling can be defensible without retraining a recognizer.

In [ ]:
KEY_COLS = ["case", "label", "left", "right", "left_id", "right_id"]
baseline = "baseline_facenet_vggface2"
upper = "upper_face_facenet_vggface2"


def stable_fraction(*parts: object) -> float:
    text = "|".join(str(part) for part in parts)
    digest = hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]
    return int(digest, 16) / float(16 ** 12)


def add_eval_split(score_df: pd.DataFrame) -> pd.DataFrame:
    out = score_df.copy()
    out["split"] = [
        "calibration" if stable_fraction(row.left, row.right, row.case, RANDOM_SEED) < 0.5 else "eval"
        for row in out.itertuples()
    ]
    return out


def summarize_score_rows(score_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, case), case_df in score_df.groupby(["model", "case"]):
        labels = case_df["label"].to_numpy()
        score_values = case_df["score"].to_numpy()
        auc = roc_auc_score(labels, score_values) if len(np.unique(labels)) == 2 else math.nan
        threshold_metrics = best_threshold_metrics(labels, score_values)
        rows.append({
            "model": model,
            "case": case,
            "pairs": len(case_df),
            "split": case_df["split"].iloc[0] if "split" in case_df else "all",
            "alpha": float(case_df["alpha"].iloc[0]) if "alpha" in case_df and case_df["alpha"].notna().any() else math.nan,
            "roc_auc": float(auc),
            **threshold_metrics,
        })
    return pd.DataFrame(rows)


def tune_alpha(calibration_df: pd.DataFrame, base_col: str, other_col: str) -> float:
    if calibration_df.empty or len(calibration_df["label"].unique()) < 2:
        return 0.5
    labels = calibration_df["label"].to_numpy()
    best_alpha = 0.5
    best_auc = -1.0
    for alpha in np.linspace(0.0, 1.0, 21):
        fused = alpha * calibration_df[base_col].to_numpy() + (1.0 - alpha) * calibration_df[other_col].to_numpy()
        auc = roc_auc_score(labels, fused)
        if auc > best_auc:
            best_auc = float(auc)
            best_alpha = float(alpha)
    return best_alpha


def build_fusion_scores(split_scores: pd.DataFrame, base_model: str, other_model: str, fusion_name: str) -> pd.DataFrame:
    wide = split_scores.pivot_table(index=KEY_COLS + ["split"], columns="model", values="score").reset_index()
    if base_model not in wide or other_model not in wide:
        return pd.DataFrame()
    wide = wide.dropna(subset=[base_model, other_model])
    calibration = wide[(wide["case"] == "masked-unmasked") & (wide["split"] == "calibration")]
    alpha = tune_alpha(calibration, base_model, other_model)
    eval_rows = wide[wide["split"] == "eval"].copy()
    eval_rows["score"] = alpha * eval_rows[base_model] + (1.0 - alpha) * eval_rows[other_model]
    eval_rows["model"] = fusion_name
    eval_rows["alpha"] = alpha
    return eval_rows[KEY_COLS + ["model", "score", "split", "alpha"]]


split_scores = add_eval_split(scores)
base_eval_scores = split_scores[split_scores["split"] == "eval"].copy()
base_eval_scores["alpha"] = math.nan

fusion_configs = [
    (baseline, "upper_face_facenet_vggface2", "fusion_full_upper_vggface2"),
    (baseline, "lower_blackout_facenet_vggface2", "fusion_full_lower_blackout_vggface2"),
    (baseline, "lower_blur_facenet_vggface2", "fusion_full_lower_blur_vggface2"),
]
fusion_scores = pd.concat(
    [build_fusion_scores(split_scores, base_model, other_model, fusion_name) for base_model, other_model, fusion_name in fusion_configs],
    ignore_index=True,
)
feasibility_scores = pd.concat([base_eval_scores[KEY_COLS + ["model", "score", "split", "alpha"]], fusion_scores], ignore_index=True)
feasibility_metrics = summarize_score_rows(feasibility_scores).sort_values(["case", "model"])

feasibility_scores.to_csv(RESULTS_DIR / "feasibility_pair_scores.csv", index=False)
feasibility_metrics.to_csv(RESULTS_DIR / "feasibility_results.csv", index=False)
display(feasibility_metrics)


In [ ]:
plt.figure(figsize=(11, 4))
sns.barplot(data=metrics, x="case", y="roc_auc", hue="model")
plt.ylim(0.0, 1.0)
plt.title("ROC-AUC by verification case")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "roc_auc_by_case.png", dpi=160)
plt.show()

display(metrics.pivot(index="model", columns="case", values="roc_auc"))

## Qualitative Differences

In [ ]:
def thresholds_by_model_case(metrics: pd.DataFrame) -> Dict[Tuple[str, str], float]:
    return {(row.model, row.case): float(row.threshold) for row in metrics.itertuples()}


def save_pair_montage(row: pd.Series, out_path: Path, title: str) -> None:
    left = Image.open(row["left"]).convert("RGB").resize((220, 220))
    right = Image.open(row["right"]).convert("RGB").resize((220, 220))
    canvas = Image.new("RGB", (440, 260), "white")
    canvas.paste(left, (0, 40))
    canvas.paste(right, (220, 40))
    draw = ImageDraw.Draw(canvas)
    draw.text((8, 8), title[:80], fill=(0, 0, 0))
    canvas.save(out_path)


thresholds = thresholds_by_model_case(metrics)
wide = scores.pivot_table(
    index=["case", "label", "left", "right", "left_id", "right_id"],
    columns="model",
    values="score",
).reset_index()

baseline = "baseline_facenet_vggface2"
upper = "upper_face_facenet_vggface2"
examples = []
for _, row in wide.iterrows():
    if baseline not in row or upper not in row or pd.isna(row[baseline]) or pd.isna(row[upper]):
        continue
    b_pred = int(row[baseline] >= thresholds[(baseline, row["case"])])
    u_pred = int(row[upper] >= thresholds[(upper, row["case"])])
    label = int(row["label"])
    if b_pred != label and u_pred == label:
        examples.append((abs(row[upper] - row[baseline]), row))

for i, (_, row) in enumerate(sorted(examples, key=lambda x: x[0], reverse=True)[:12], start=1):
    title = f"{row['case']} label={row['label']} baseline={row[baseline]:.3f} upper={row[upper]:.3f}"
    save_pair_montage(row, QUAL_DIR / f"upper_beats_baseline_{i:02d}.jpg", title)

print(f"Saved {min(len(examples), 12)} qualitative examples to {QUAL_DIR}")

## Go / No-Go Recommendation

In [ ]:
def metric_value(metrics: pd.DataFrame, model: str, case: str, metric: str) -> float:
    value = metrics[(metrics["model"] == model) & (metrics["case"] == case)][metric]
    return float(value.iloc[0]) if len(value) else math.nan


conclusion_metrics = feasibility_metrics if "feasibility_metrics" in globals() and not feasibility_metrics.empty else metrics
lightweight_models = [
    model for model in conclusion_metrics["model"].unique()
    if model != baseline and not str(model).startswith("masked_specific_candidate")
]
primary_case = conclusion_metrics[conclusion_metrics["case"] == "masked-unmasked"]
candidate_rows = primary_case[primary_case["model"].isin(lightweight_models)]
best_lightweight = candidate_rows.sort_values("roc_auc", ascending=False).iloc[0].model if len(candidate_rows) else upper

baseline_mu_auc = metric_value(conclusion_metrics, baseline, "masked-unmasked", "roc_auc")
upper_mu_auc = metric_value(conclusion_metrics, upper, "masked-unmasked", "roc_auc")
best_mu_auc = metric_value(conclusion_metrics, best_lightweight, "masked-unmasked", "roc_auc")
baseline_uu_auc = metric_value(conclusion_metrics, baseline, "unmasked-unmasked", "roc_auc")
best_uu_auc = metric_value(conclusion_metrics, best_lightweight, "unmasked-unmasked", "roc_auc")

masked_unmasked_gain = best_mu_auc - baseline_mu_auc
unmasked_regression = baseline_uu_auc - best_uu_auc
feasible = masked_unmasked_gain >= 0.02 and unmasked_regression <= 0.03
recommendation = "FEASIBLE" if feasible else "NOT YET FEASIBLE"

conclusion = f"""# Validation Spike Conclusion

Recommendation: {recommendation}

- Question: can lightweight occlusion handling be a defensible course-project direction without retraining?
- Best lightweight model: {best_lightweight}
- Masked-unmasked ROC-AUC baseline: {baseline_mu_auc:.4f}
- Masked-unmasked ROC-AUC upper-face: {upper_mu_auc:.4f}
- Masked-unmasked ROC-AUC best lightweight: {best_mu_auc:.4f}
- Masked-unmasked gain vs baseline: {masked_unmasked_gain:.4f}
- Unmasked-unmasked ROC-AUC baseline: {baseline_uu_auc:.4f}
- Unmasked-unmasked ROC-AUC best lightweight: {best_uu_auc:.4f}
- Unmasked regression vs baseline: {unmasked_regression:.4f}

Feasibility rule: promising only if a lightweight method improves masked-unmasked ROC-AUC by at least 0.02 while dropping unmasked-unmasked ROC-AUC by no more than 0.03.

Third model note: {MASKED_SPECIFIC_MODEL_NOTE}
"""
(RESULTS_DIR / "validation_conclusion.md").write_text(conclusion)
print(conclusion)